<a href="https://colab.research.google.com/github/m1Febriansyah/244107020199-ML-12/blob/main/JS11Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Membaca Dataset
df = pd.read_csv('voice (1).csv')

# 2. Preprocessing: Encoding Label dan Scaling Fitur
le = LabelEncoder()
y = le.fit_transform(df['label']) # male=1, female=0
X = df.drop('label', axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Pengujian Berbagai Rasio Split dan Kernel
rasio_split = {'70:30': 0.3, '80:20': 0.2}
kernels = ['linear', 'poly', 'rbf']
hasil_evaluasi = []

for nama_rasio, test_size in rasio_split.items():
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=test_size, random_state=42)

    for k in kernels:
        # Pembuatan dan Pelatihan Model
        model = SVC(kernel=k)
        model.fit(X_train, y_train)

        # Prediksi dan Akurasi
        y_pred = model.predict(X_test)
        akurasi = accuracy_score(y_test, y_pred)

        # Simpan hasil
        hasil_evaluasi.append({
            'Rasio Split': nama_rasio,
            'Kernel': k.capitalize(),
            'Akurasi (%)': round(akurasi * 100, 2)
        })

# 4. Tabulasi Hasil
df_hasil = pd.DataFrame(hasil_evaluasi)
print("=== TABULASI PERFORMANSI SVM (VOICE.CSV) ===")
print(df_hasil.to_string(index=False))

=== TABULASI PERFORMANSI SVM (VOICE.CSV) ===
Rasio Split Kernel  Akurasi (%)
      70:30 Linear        97.06
      70:30   Poly        95.69
      70:30    Rbf        98.11
      80:20 Linear        97.63
      80:20   Poly        96.85
      80:20    Rbf        98.26


In [9]:
import os

zip_file = 'images.zip'

if os.path.exists(zip_file):
    # 1. Cek ukuran file
    size_bytes = os.path.getsize(zip_file)
    print(f"Ukuran berkas {zip_file}: {size_bytes / (1024*1024):.2f} MB ({size_bytes} bytes)")

    # 2. Cek tipe berkas menggunakan perintah Linux
    print("Tipe berkas berdasarkan sistem:")
    !file images.zip

    # 3. Ekstrak menggunakan command line system unzip
    print("\nMengekstrak menggunakan perintah sistem...")
    !unzip -q -o images.zip -d .

    # 4. Validasi isi folder jika berhasil
    if os.path.exists('images'):
        print("\nEkstraksi Berhasil! Folder 'images/' terdeteksi:")
        !ls -d images/*
    else:
        print("\nEkstraksi gagal. Folder 'images/' tidak ditemukan.")
else:
    print(f"File '{zip_file}' tidak ditemukan di direktori Colab Anda. Silakan unggah kembali berkas tersebut.")

Ukuran berkas images.zip: 27.51 MB (28845089 bytes)
Tipe berkas berdasarkan sistem:
images.zip: Zip archive data, at least v2.0 to extract, compression method=store

Mengekstrak menggunakan perintah sistem...

Ekstraksi Berhasil! Folder 'images/' terdeteksi:
images/test  images/training


In [10]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

def extract_hsv_histogram(img_path):
    """
    Mengekstrak fitur histogram 3-channel (H, S, V) dari gambar
    dan menggabungkannya menjadi satu vektor fitur 1D.
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return None

    # Konversi BGR ke HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Hitung histogram untuk tiap channel (Hue, Saturation, Value)
    # Digunakan 32 bins per channel untuk mereduksi dimensi fitur
    hist_h = cv2.calcHist([hsv], [0], None, [32], [0, 180])
    hist_s = cv2.calcHist([hsv], [1], None, [32], [0, 256])
    hist_v = cv2.calcHist([hsv], [2], None, [32], [0, 256])

    # Gabungkan (concatenate) ketiga histogram dan jadikan array 1D
    hist_features = np.concatenate([hist_h, hist_s, hist_v]).flatten()

    # Normalisasi agar nilai fitur tidak terpengaruh resolusi/ukuran gambar
    cv2.normalize(hist_features, hist_features)

    return hist_features

# 1. Load Data dan Ekstraksi Fitur Histogram
base_dir = Path('images') # Sesuaikan path ini dengan folder ekstraksi zip Anda
fitur_list = []
label_list = []

print("Mengekstrak fitur histogram citra...")
# Mengambil semua file gambar di dalam subfolder 'day' dan 'night'
for file_path in base_dir.rglob('*.jpg'):
    label_folder = file_path.parent.name.lower()

    if label_folder in ['day', 'night']:
        hist_feat = extract_hsv_histogram(file_path)
        if hist_feat is not None:
            fitur_list.append(hist_feat)
            label_list.append(1 if label_folder == 'day' else 0)

X = np.array(fitur_list)
y = np.array(label_list)

# 2. Split Data Rasio 80:20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Hyperparameter Tuning menggunakan GridSearchCV (Kernel RBF)
print("\nMelakukan hyperparameter tuning untuk RBF...")
param_grid = {
    'C': [0.1, 1, 10, 100],           # Parameter regulariasi
    'gamma': ['scale', 'auto', 0.01, 0.1, 1] # Koefisien kernel RBF
}

# Inisialisasi model dan GridSearch
svm_rbf = SVC(kernel='rbf')
grid_search = GridSearchCV(svm_rbf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Latih untuk mencari hyperparameter terbaik
grid_search.fit(X_train, y_train)

# 4. Evaluasi Performa
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
akurasi_final = accuracy_score(y_test, y_pred)

print("=== HASIL KLASIFIKASI SIANG/MALAM (SVM HISTOGRAM) ===")
print(f"Hyperparameter Terbaik : {grid_search.best_params_}")
print(f"Akurasi Data Uji (80:20): {akurasi_final * 100:.2f} %")
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred, target_names=['Night', 'Day']))

Mengekstrak fitur histogram citra...

Melakukan hyperparameter tuning untuk RBF...
=== HASIL KLASIFIKASI SIANG/MALAM (SVM HISTOGRAM) ===
Hyperparameter Terbaik : {'C': 10, 'gamma': 'scale'}
Akurasi Data Uji (80:20): 100.00 %

Laporan Klasifikasi:
               precision    recall  f1-score   support

       Night       1.00      1.00      1.00        42
         Day       1.00      1.00      1.00        38

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80

